In [43]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

### Работа со skills

Анализ навыков для сайта [hirify](https://hirify.me).

In [44]:
data = pd.read_csv('vacancies_cleaned.csv')
data.sample(3)

,title,link,tags,company,skills,salary,relocation,employment_type,onsite,remote,hybrid,lead,c_level,senior,director,trainee,head,middle,junior,location
2565,Tech Lead Manager (Client Integrations),https://hirify.me/jobs/438686-tech-lead-manage...,"['remote (Europe)', 'Europe', 'fulltime', 'lead']",Company hidden,"['ai', 'python', 'llm', 'distributed systems',...",140 000 - 185 000\n$,False,fulltime,True,False,True,True,False,False,False,False,False,False,False,Europe
5980,Финансовый директор (Нефтегаз),https://hirify.me/jobs/415974-zamestitel-gener...,"['onsite', 'Russia', 'fulltime', 'c_level']",NaN,"['budgeting', 'treasury', 'ifrs', 'cpa', 'sap'...",NaN,False,fulltime,False,True,True,False,True,False,False,False,False,False,False,Russia
7171,Middle Backend Developer,https://hirify.me/jobs/433149-middle-backend-d...,"['hybrid', 'Russia', 'fulltime', 'middle']",NaN,"['python', 'redis', 'rabbitmq', 'fastapi', 'do...",130 000\n₽,False,fulltime,True,True,False,False,False,False,False,False,False,True,False,Russia


Перед началом обработки определим дополнительный признак - количество требований вакансии.

In [65]:
def get_correct_len(array):
    l = 0
    if not array:
        return 0
    for value in array:
        if ' skills' in value and value[0] == '+':
            l += int(value.split(' ')[0])
        else:
            l += 1
    return l

In [67]:
data['skills'][3]

['communication skills',
 'ai',
 'customer service',
 'leadership',
 'technical support',
 '+2 skills']

In [66]:
get_correct_len(data['skills'][3])

7

Данные в колонке представлены строковым типом. для дальнейшей обработки преобразуем в массив с помощью функции: 

In [45]:
def str_to_list(skills_string):

    try:
        if not skills_string or skills_string == "[]":
            return []
        skills_string = skills_string[1:-1].split(",")
        skills_list = [
            skill.replace("'", "").strip().lower() for skill in skills_string
        ]
        skills_list = set(skills_list)
        return list(skills_list)

    except Exception:
        return []

Для формирования дополнительного столбца используется метод [`explode`](https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.explode.html):

In [46]:
data["skills"] = data["skills"].apply(lambda x: str_to_list(x))
data_skills = (
    data.reset_index()[["index", "skills"]].explode("skills").rename(columns={"skills": "skill", "index": 'id'})
)
data_skills

,id,skill
0,0,sales
0,0,healthcare
0,0,saas
0,0,negotiation
0,0,crm
...,...,...
10552,10552,sql
10552,10552,customer support
10552,10552,databases
10552,10552,excel


In [47]:
data_skills['skill'].value_counts().head(10)

skill
ai           3190
+3 skills    2241
+2 skills    2167
python       1718
+4 skills    1639
+1 skills    1381
+5 skills    1017
sql           979
saas          920
fintech       880
Name: count, dtype: int64

In [51]:
print(f'Количество пропусков: {data_skills['skill'].isna().sum()}')

Количество пропусков: 0


In [49]:
data_skills = data_skills.dropna()

In [ ]:
data_skills[data_skills['skill'].str.contains(r"\+")] = "No "

,id,skill
0,0,+2 skills
1,1,+1 skills
2,2,+10 skills
3,3,+2 skills
4,4,+1 skills
...,...,...
10548,10548,+4 skills
10549,10549,+2 skills
10550,10550,+3 skills
10551,10551,+3 skills
